In [ ]:
##############################################################
# Script to Build the Instruction Prompt Text (with timestamp)
##############################################################

In [55]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
import json
import json
import random
from pydantic import BaseModel
from pydantic import ValidationError
import json
import datetime as dt

In [56]:
def create_prompts(papers, paper_id, text_type="summary"):
    """
    Given a list of paper records (as dictionaries) and a specific paper_id,
    returns a dictionary with the keys:
        {
          "paper_id": "paper_id",
          "messages": [
            { "role": "system", "content": "...system content..." },
            { "role": "user", "content": "...user content..." },
            { "role": "assistant", "content": "...assistant content..." }
          ]
        }
    which can be used as a single training example for instruction fine-tuning.

    Parameters:
      papers (list): A list of dictionaries, each representing a paper.
      paper_id (str): The ID of the desired paper to create a prompt for.
      text_type (str): Which field to display in the user prompt.
                      Options: "summary", "abstract", "full text".
                      Default is "summary".
    """

    paper_data = next((p for p in papers if p["paper_id"] == paper_id), None)
    if not paper_data:
        raise ValueError(f"No paper found with paper_id={paper_id}")

    if text_type not in ("summary", "abstract", "full text"):
        raise ValueError("text_type must be one of: 'summary', 'abstract', 'full text'")


    label_map = {
        "summary": "PAPER SUMMARY",
        "abstract": "PAPER ABSTRACT",
        "full text": "PAPER"
    }
    text_label = label_map[text_type]


    date_str = paper_data.get("date", "")
    try:
        parsed_date = dt.datetime.strptime(date_str, "%m-%d-%Y")
        year_month = f"{parsed_date.year}-{parsed_date.month:02d}"
    except ValueError:
        year_month = "0000-00"

    # 4) System message
    system_message = (
        "You are a helpful assistant that extracts a structured summary from a paper. "
        "Please return only valid JSON with the following fields:\n"
        "Soundness, Presentation, Contribution, Rating, Confidence (all integers)\n"
        "Strengths, Weaknesses, and Questions (all strings).\n"
        "No additional keys. No extra text."
    )

    user_message = (
        f"Please read the following {text_label.lower()} and produce reviewer scores.\n\n"
        f"--- {text_label} ---\n\n"
        f"<|year_month={year_month}|>\n"
        f"{paper_data[text_type]}\n\n"
        f"--- REQUIRED CATEGORIES ---\n"
        f"We need the following fields in the output:\n"
        f"1) Soundness\n"
        f"2) Presentation\n"
        f"3) Contribution\n"
        f"4) Rating\n"
        f"5) Confidence\n"
        f"6) Strengths\n"
        f"7) Weaknesses\n"
        f"8) Questions\n\n"
        f"--- POSSIBLE VALUES & DEFINITIONS ---\n\n"
        f"Soundness, Presentation, and Contribution (1–4):\n"
        f"- 1: poor\n"
        f"- 2: fair\n"
        f"- 3: good\n"
        f"- 4: excellent\n\n"
        f"Rating (1–10):\n"
        f"- 1: Trivial or wrong\n"
        f"- 2: Strong rejection\n"
        f"- 3: Clear rejection\n"
        f"- 4: Ok but not good enough - rejection\n"
        f"- 5: Marginally below acceptance threshold\n"
        f"- 6: Marginally above acceptance threshold\n"
        f"- 7: Good paper, accept\n"
        f"- 8: Top 50% of accepted papers, clear accept\n"
        f"- 9: Top 15% of accepted papers, strong accept\n"
        f"- 10: Top 5% of accepted papers, seminal paper\n\n"
        f"Confidence (1–5):\n"
        f"- 1: The reviewer's evaluation is an educated guess\n"
        f"- 2: The reviewer is willing to defend the evaluation, but might be wrong\n"
        f"- 3: The reviewer is fairly confident that the evaluation is correct\n"
        f"- 4: The reviewer is confident but not absolutely certain that the evaluation is correct\n"
        f"- 5: The reviewer is absolutely certain that the evaluation is correct and very familiar with the topic\n\n"
        f"--- OUTPUT FORMAT ---\n"
        f"Please produce only a JSON object with this exact structure (no extra keys):\n"
        f"{{\n"
        f"  \"Soundness\": <int>,\n"
        f"  \"Presentation\": <int>,\n"
        f"  \"Contribution\": <int>,\n"
        f"  \"Rating\": <int>,\n"
        f"  \"Confidence\": <int>,\n"
        f"  \"Strengths\": \"<string>\",\n"
        f"  \"Weaknesses\": \"<string>\",\n"
        f"  \"Questions\": \"<string>\"\n"
        f"}}\n\n"
        f"No explanations or markdown formatting. Only valid JSON."
    )

    assistant_content = json.dumps(paper_data["response"], ensure_ascii=False)

    return {
        "paper_id": paper_id,
        "messages": [
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            },
            {
                "role": "assistant",
                "content": assistant_content
            }
        ]
    }

In [67]:
with open("data/test_data/test_data_without_prompt/test_data_2025_timestamp.json", "r", encoding="utf-8") as f:
    papers_data = json.load(f)

papers_data[:2]

[{'paper_id': 'aAcOaJYbUg',
  'full text': '\nLAION-C: AN OUT-OF-DISTRIBUTION\nBENCHMARK FOR WEB-SCALE VISION MODELS\n\nAnonymous authors\nPaper under double-blind review\n\nABSTRACT\n\nOut-of-distribution (OOD) robustness is a desired property of computer vision\nmodels. Improving model robustness requires high-quality signals from robust-\nness benchmarks to quantify progress. While various benchmark datasets such\nas ImageNet-C were proposed in the ImageNet era, most ImageNet-C corruption\ntypes are no longer OOD relative to today’s large datasets scraped from the web,\nwhich already contain common corruptions such as blur or JPEG compression\nartifacts. Consequently, these standard benchmarks are no longer well-suited for\nevaluating OOD robustness in the era of web-scale datasets. Indeed, recent mod-\nels show saturating scores on ImageNet-era OOD benchmarks, indicating that it\nis unclear whether models trained on web-scale datasets truly become better at\nOOD generalization or w

In [68]:
# OPTIONAL ONLY IF REFORMATTING NEEDED

def convert_dates(papers):
    """
    Convert the 'date' field in each paper dictionary from
    "YYYY-MM-DD HH:MM:SS.microseconds+00:00" format to "MM-DD-YYYY" format.

    Args:
        papers (list): List of dictionaries where each dictionary represents a paper.

    Returns:
        list: The list with updated date formats.
    """
    for paper in papers:
        original_date = paper.get('date', '')
        if original_date:
            # Parse the ISO 8601 date string
            date_obj = dt.datetime.fromisoformat(original_date)
            # Format the datetime object to "MM-DD-YYYY"
            paper['date'] = date_obj.strftime("%m-%d-%Y")
    return papers


In [69]:
papers_data = convert_dates(papers_data)
papers_data[:2]

[{'paper_id': 'aAcOaJYbUg',
  'full text': '\nLAION-C: AN OUT-OF-DISTRIBUTION\nBENCHMARK FOR WEB-SCALE VISION MODELS\n\nAnonymous authors\nPaper under double-blind review\n\nABSTRACT\n\nOut-of-distribution (OOD) robustness is a desired property of computer vision\nmodels. Improving model robustness requires high-quality signals from robust-\nness benchmarks to quantify progress. While various benchmark datasets such\nas ImageNet-C were proposed in the ImageNet era, most ImageNet-C corruption\ntypes are no longer OOD relative to today’s large datasets scraped from the web,\nwhich already contain common corruptions such as blur or JPEG compression\nartifacts. Consequently, these standard benchmarks are no longer well-suited for\nevaluating OOD robustness in the era of web-scale datasets. Indeed, recent mod-\nels show saturating scores on ImageNet-era OOD benchmarks, indicating that it\nis unclear whether models trained on web-scale datasets truly become better at\nOOD generalization or w

In [70]:
# Create a training example for paper_id, using the "abstract" field instead of the default "summary"
example = create_prompts(papers_data, paper_id="aAcOaJYbUg", text_type="abstract")
messages_for_prompt = example["messages"]
prompt_messages = messages_for_prompt[:2]
prompt_messages

[{'role': 'system',
  'content': 'You are a helpful assistant that extracts a structured summary from a paper. Please return only valid JSON with the following fields:\nSoundness, Presentation, Contribution, Rating, Confidence (all integers)\nStrengths, Weaknesses, and Questions (all strings).\nNo additional keys. No extra text.'},
 {'role': 'user',
  'content': 'Please read the following paper abstract and produce reviewer scores.\n\n--- PAPER ABSTRACT ---\n\n<|year_month=2024-09|>\nOut-of-distribution (OOD) robustness is a desired property of computer vision models. Improving model robustness requires high-quality signals from robustness benchmarks to quantify progress. While various benchmark datasets such as ImageNet-C were proposed in the ImageNet era, most ImageNet-C corruption types are no longer OOD relative to today\'s large datasets scraped from the web, which already contain common corruptions such as blur or JPEG compression artifacts. Consequently, these standard benchmark

In [71]:
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

class PaperReview(BaseModel):
    Soundness: int
    Presentation: int
    Contribution: int
    Rating: int
    Confidence: int
    Strengths: str
    Weaknesses: str
    Questions: str

In [62]:
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=prompt_messages
)

raw_content = completion.choices[0].message.content

try:
    parsed_dict = json.loads(raw_content)
    review = PaperReview(**parsed_dict)
    print(review)
except json.JSONDecodeError as e:
    print("Invalid JSON returned by the model:", e)
except ValidationError as ve:
    print("Pydantic validation error:", ve)


Invalid JSON returned by the model: Expecting value: line 1 column 1 (char 0)


In [63]:
raw_content

'```json\n{\n  "Soundness": 3,\n  "Presentation": 3,\n  "Contribution": 3,\n  "Rating": 7,\n  "Confidence": 4,\n  "Strengths": "The paper proposes a novel framework, PAPG, that effectively combines reinforcement learning with supervised learning to address partially annotated multi-label classification challenges. The introduction of local and global rewards and an iterative training strategy with data enhancement are innovative contributions.",\n  "Weaknesses": "The paper does not sufficiently address the potential computational complexity of the proposed PAPG framework. Additionally, the effectiveness of the approach in scenarios significantly different from the ones tested is not discussed, potentially limiting generalizability.",\n  "Questions": "How does the computational complexity of PAPG compare with existing methods for multi-label classification? Are any specific computational optimizations used to handle large-scale datasets?"\n}\n```'

In [72]:
papers_data

[{'paper_id': 'aAcOaJYbUg',
  'full text': '\nLAION-C: AN OUT-OF-DISTRIBUTION\nBENCHMARK FOR WEB-SCALE VISION MODELS\n\nAnonymous authors\nPaper under double-blind review\n\nABSTRACT\n\nOut-of-distribution (OOD) robustness is a desired property of computer vision\nmodels. Improving model robustness requires high-quality signals from robust-\nness benchmarks to quantify progress. While various benchmark datasets such\nas ImageNet-C were proposed in the ImageNet era, most ImageNet-C corruption\ntypes are no longer OOD relative to today’s large datasets scraped from the web,\nwhich already contain common corruptions such as blur or JPEG compression\nartifacts. Consequently, these standard benchmarks are no longer well-suited for\nevaluating OOD robustness in the era of web-scale datasets. Indeed, recent mod-\nels show saturating scores on ImageNet-era OOD benchmarks, indicating that it\nis unclear whether models trained on web-scale datasets truly become better at\nOOD generalization or w

In [73]:
parameter_list = ["abstract"]

for parameter in parameter_list:
    train_examples = []
    for p in papers_data:
        example = create_prompts(papers_data, p["paper_id"], text_type=parameter)
        train_examples.append(example)
    with open(f"data/test_data/test_data_with_timestamps/test_data_2025_{parameter}_prompts_with_timestamp.jsonl", "w", encoding="utf-8") as out_f:
        for ex in train_examples:
            out_f.write(json.dumps(ex) + "\n")

In [23]:
random.seed(42)
train_subset = random.sample(papers_data, 100)

remaining_for_test = [p for p in papers_data if p not in train_subset]
test_subset = random.sample(remaining_for_test, 100)
test_subset_2025 = random.sample(papers_data_2025, 100)


In [24]:
# run everything for summary, abstract, and full text
parameter_list = ["full text"]

for parameter in parameter_list:
    train_examples = []
    for p in train_subset:
        example = create_prompts(papers_data, p["paper_id"], text_type=parameter)
        train_examples.append(example)

    test_examples = []
    for p in test_subset:
        example = create_prompts(papers_data, p["paper_id"], text_type=parameter)
        test_examples.append(example)
    with open(f"data/training_data_2024_{parameter}_prompts.jsonl", "w", encoding="utf-8") as out_f:
        for ex in train_examples:
            out_f.write(json.dumps(ex) + "\n")

    with open(f"data/test_data_2024_{parameter}_prompts.jsonl", "w", encoding="utf-8") as out_f:
        for ex in test_examples:
            out_f.write(json.dumps(ex) + "\n")

    test_examples_2025 = []
    for p in test_subset_2025:
        example = create_prompts(papers_data_2025, p["paper_id"], text_type=parameter)
        test_examples_2025.append(example)

    with open(f"data/test_data_2025_{parameter}_prompts.jsonl", "w", encoding="utf-8") as out_f:
        for ex in test_examples_2025:
            out_f.write(json.dumps(ex) + "\n")

In [25]:
# get remaining data for 2024 and 2025
remaining_data_2024 = [p for p in papers_data if p not in train_subset and p not in test_subset]
remaining_data_2025 = [p for p in papers_data_2025 if p not in test_subset_2025]

In [26]:
# output remaining data to json
with open("data/unused_data_2024.json", "w", encoding="utf-8") as out_f:
    json.dump(remaining_data_2024, out_f, ensure_ascii=False, indent=2)

with open("data/unused_data_2025.json", "w", encoding="utf-8") as out_f:
    json.dump(remaining_data_2025, out_f, ensure_ascii=False, indent=2)

In [28]:
# run everything for summary, abstract, and full text of unused data
for parameter in parameter_list:
    # Create training examples
    train_examples = []
    for p in remaining_data_2024:
        example = create_prompts(papers_data, p["paper_id"], text_type=parameter)
        train_examples.append(example)

    # Save training examples to JSONL
    with open(f"data/temp_unused_data/unused_data_2024_{parameter}_prompts.jsonl", "w", encoding="utf-8") as out_f:
        for ex in train_examples:
            out_f.write(json.dumps(ex) + "\n")

    # do the same for 2025
    train_examples = []
    for p in remaining_data_2025:
        example = create_prompts(papers_data_2025, p["paper_id"], text_type=parameter)
        train_examples.append(example)

    # Save training examples to JSONL
    with open(f"data/temp_unused_data/unused_data_2025_{parameter}_prompts.jsonl", "w", encoding="utf-8") as out_f:
        for ex in train_examples:
            out_f.write(json.dumps(ex) + "\n")



In [44]:
def load_jsonl(filepath):
    """
    Loads a JSONL file (each line is a JSON object).
    Returns a list of dictionaries.
    """
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

def save_jsonl(data, filepath):
    """
    Saves a list of dictionaries to a JSONL file, one line per dict.
    """
    with open(filepath, "w", encoding="utf-8") as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [49]:
def get_example_prompt(data_list, paper_id):
    """
    Finds the record in data_list with the specified paper_id.
    Returns (example_user_message, example_assistant_message) if found.
    """
    for item in data_list:
        if item.get("paper_id") == paper_id:
            user_msg = None
            assistant_msg = None
            for msg in item.get("messages", []):
                if msg["role"] == "user":
                    user_msg = msg["content"]
                elif msg["role"] == "assistant":
                    assistant_msg = msg["content"]
            if user_msg and assistant_msg:
                return (user_msg, assistant_msg)
    return (None, None)


def build_prefix_from_examples(example_list):
    """
    Given a list of (user, assistant) examples, build a text block
    that shows them in sequence for few-shot demonstration.

    Example output structure:
      Below are example prompts & responses:

      EXAMPLE 1 (USER):
      ...

      EXAMPLE 1 (ASSISTANT):
      ...
      -----

      ...
      NOW PLEASE DO THE SAME FOR THIS NEW PAPER BELOW.
    """
    if not example_list:
        return ""  # No examples => no prefix

    lines = ["Below are example prompts & responses:\n"]
    for i, (u, a) in enumerate(example_list, start=1):
        lines.append(f"EXAMPLE {i} (USER):\n{u}\n")
        lines.append(f"EXAMPLE {i} (ASSISTANT):\n{a}\n")
        lines.append("-----\n")
    lines.append("NOW PLEASE DO THE SAME FOR THIS NEW PAPER BELOW.\n")

    return "\n".join(lines)

In [50]:
def rebuild_user_message(original_user_text, prefix):
    """
    Prepend prefix (few-shot/one-shot examples) to the user message text.
    """
    if not prefix:
        return original_user_text
    return prefix + "\n" + original_user_text

In [52]:
def create_shot_versions(
    example_2024_file,
    input_file_2024,
    input_file_2025,
    output_file_2024_one,
    output_file_2024_few,
    output_file_2025_one,
    output_file_2025_few,
    one_shot_ids,
    few_shot_ids
):
    """
    1) Load the "example_2024_file" to retrieve your demonstration examples.
    2) Build a one-shot prefix (from one_shot_ids) and a few-shot prefix (from few_shot_ids).
    3) Load the main 2024 and 2025 data, prepend the prefixes to each user message.
    4) Save the resulting data to new JSONL files for one-shot / few-shot.
    """

    # Step A: Load the example file (2024) to find demonstration examples
    examples_2024 = load_jsonl(example_2024_file)

    one_shot_examples = []
    for pid in one_shot_ids:
        u_msg, a_msg = get_example_prompt(examples_2024, pid)
        if u_msg and a_msg:
            one_shot_examples.append((u_msg, a_msg))

    few_shot_examples = []
    for pid in few_shot_ids:
        u_msg, a_msg = get_example_prompt(examples_2024, pid)
        if u_msg and a_msg:
            few_shot_examples.append((u_msg, a_msg))

    one_shot_prefix = build_prefix_from_examples(one_shot_examples)
    few_shot_prefix = build_prefix_from_examples(few_shot_examples)

    data_2024 = load_jsonl(input_file_2024)
    data_2025 = load_jsonl(input_file_2025)

    new_2024_one_shot = []
    for item in data_2024:
        item_copy = json.loads(json.dumps(item))
        for msg in item_copy["messages"]:
            if msg["role"] == "user":
                msg["content"] = rebuild_user_message(msg["content"], one_shot_prefix)
        new_2024_one_shot.append(item_copy)

    new_2024_few_shot = []
    for item in data_2024:
        item_copy = json.loads(json.dumps(item))
        for msg in item_copy["messages"]:
            if msg["role"] == "user":
                msg["content"] = rebuild_user_message(msg["content"], few_shot_prefix)
        new_2024_few_shot.append(item_copy)

    new_2025_one_shot = []
    for item in data_2025:
        item_copy = json.loads(json.dumps(item))
        for msg in item_copy["messages"]:
            if msg["role"] == "user":
                msg["content"] = rebuild_user_message(msg["content"], one_shot_prefix)
        new_2025_one_shot.append(item_copy)

    new_2025_few_shot = []
    for item in data_2025:
        item_copy = json.loads(json.dumps(item))
        for msg in item_copy["messages"]:
            if msg["role"] == "user":
                msg["content"] = rebuild_user_message(msg["content"], few_shot_prefix)
        new_2025_few_shot.append(item_copy)

    save_jsonl(new_2024_one_shot, output_file_2024_one)
    save_jsonl(new_2024_few_shot, output_file_2024_few)
    save_jsonl(new_2025_one_shot, output_file_2025_one)
    save_jsonl(new_2025_few_shot, output_file_2025_few)

    print("Done! Created one-shot & few-shot versions.")

In [62]:
for parameter in parameter_list:

    example_2024_file = f"data/example_data/unused_data_2024_{parameter}_prompts.jsonl"

    # We'll transform these two sets:
    input_file_2024 = f"data/test_data/zero_shot/test_data_2024_{parameter}_prompts.jsonl"
    input_file_2025 = f"data/test_data/zero_shot/test_data_2025_{parameter}_prompts.jsonl"

    # Output files:
    out_2024_one = f"data/test_data/one_shot/test_data_2024_{parameter}_prompts_oneshot.jsonl"
    out_2024_few = f"data/test_data/few_shot/test_data_2024_{parameter}_prompts_fewshot.jsonl"
    out_2025_one = f"data/test_data/one_shot/test_data_2025_{parameter}_prompts_oneshot.jsonl"
    out_2025_few = f"data/test_data/few_shot/test_data_2025_{parameter}_prompts_fewshot.jsonl"

    # The one/few shot IDs (all exist in the example_2024_file)
    one_shot_ids = ["o1TKGCrSL7"]
    few_shot_ids = ["o1TKGCrSL7", "JzvIWvC9MG", "5rrYpa2vts"]

    create_shot_versions(
            example_2024_file=example_2024_file,
            input_file_2024=input_file_2024,
            input_file_2025=input_file_2025,
            output_file_2024_one=out_2024_one,
            output_file_2024_few=out_2024_few,
            output_file_2025_one=out_2025_one,
            output_file_2025_few=out_2025_few,
            one_shot_ids=one_shot_ids,
            few_shot_ids=few_shot_ids
        )

Done! Created one-shot & few-shot versions.
Done! Created one-shot & few-shot versions.
Done! Created one-shot & few-shot versions.
